In [26]:
cd /Users/karolinegriesbach/Documents/Innkeepr/Git/evaluation-and-execution-scripts/

In [27]:
import logging
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import awswrangler as wr
from general_functions.datetime_helper import transform_date_to_timestamp_milliseconds
from general_functions.return_workspace_ids import return_workspace_ids
from general_functions.constants import return_api_url
from general_functions.call_api_with_account_id import call_api_with_accountId, send_to_innkeepr_api_paginated
from general_functions.conncet_s3 import S3Connection

In [28]:
customer = "More"
date = '20260531'
url = return_api_url()
print(f"url = {url}")
workspace_ids = return_workspace_ids()
workspace_id = [acc["id"] for acc in workspace_ids if acc["name"] == customer]
workspace_id = workspace_id[0]

In [29]:
s3 = S3Connection()
files = s3.list_files_with_pagination(bucket_name=workspace_id, prefix=f"firehose/events/{date}/")
print(f"Found {len(files)} files")

In [30]:
goals = call_api_with_accountId(f"{url}/goals/query", workspace_id, {}, logger=logging)
conversionEvents = [goal['conversionEvents'] for goal in goals]
conversionEvents = list(set([item for sublist in conversionEvents for item in sublist]))
conversionEvents

In [31]:
import json
import gzip
import boto3

s3 = boto3.client('s3')

data = []
list_types=["page","track"]
selected_events = []
used_types = []
track_names = []
for ifile, file in enumerate(files):
    print(f"{ifile}/{len(files)}")
    response = s3.get_object(Bucket=workspace_id, Key=file)
    raw = response['Body'].read()
    try:
        content = gzip.decompress(raw).decode('utf-8')
    except OSError:
        content = raw.decode('utf-8')
    records = [json.loads(line) for line in content.strip().splitlines() if line]
    for dat in records:
        if sorted(track_names) == sorted(conversionEvents) and sorted(list_types) == sorted(used_types):
            print(f"found everything")
            break
        name = dat.get('name', None)
        #print(dat['type'], track_names, used_types, name)
        if not dat['type']:
            continue
        if dat['type'] in used_types:
            if dat['type'] == 'track' and dat['name'] in track_names:
                #print("continue for track: ", name)
                continue
            elif dat['type'] in used_types and dat['type'] != 'track':
                #print("continue for other")
                continue
        used_types.append(dat['type'])
        if dat['type'] == 'track':
            track_names.append(dat['name'])
        if dat['type'] == 'page' and dat['meta']['referrer']=='':
            continue
        if dat['type'] in used_types:
            selected_events.append(dat)
            used_types.append(dat['type'])
        used_types = list(set(used_types))
    with open(f'DataChecks/firehose_events/{customer}_{date}_example_firehose_events.json', 'w') as f:
        json.dump(selected_events, f, indent=4)

In [32]:
with open(f'DataChecks/firehose_events/{customer}_{date}_example_firehose_events.json', 'w') as f:
        json.dump(selected_events, f, indent=4)